# Smart deduplication: merge only true duplicates by mean

- same doc_id + assay_id → merge by mean (true duplicates)
- different doc_id or assay_id → keep all (independent measurements)

In [1]:
import pandas as pd
import os
import numpy as np
from rdkit import Chem

data_dir = r'../data/in_vitro_data'
output_dir = r'../data/in_vitro_data_deduplicated'
os.makedirs(output_dir, exist_ok=True)

files = [
    'chembl_CL_raw.csv',
    'chembl_Fu_raw.csv',
    'biogen_CL_raw.csv',
    'chembl_PPB_raw.csv',
    'chembl_T1_2_raw.csv',
    'tdc_cl_hep.csv',
    'tdc_cl_mic.csv',
    'tdc_half_life.csv',
    'tdc_PPBR_AZ.csv'
]

smiles_column_candidates = ['smiles', 'SMILES', 'Smiles', 'Drug']
value_column_candidates = ['value', 'Y']

def get_canonical_smiles(smiles_str):
    if pd.isna(smiles_str):
        return None
    try:
        mol = Chem.MolFromSmiles(str(smiles_str))
        if mol is not None:
            return Chem.MolToSmiles(mol)
        else:
            return None
    except:
        return None

print("Smart Deduplication Processing...\n")

summary_data = []

Smart Deduplication Processing...



In [2]:
for file in files:
    file_path = os.path.join(data_dir, file)
    
    try:
        df = pd.read_csv(file_path)
        original_rows = len(df)
        
        # Find the SMILES column
        smiles_col = None
        for col in smiles_column_candidates:
            if col in df.columns:
                smiles_col = col
                break
        
        # Find the value column
        value_col = None
        for col in value_column_candidates:
            if col in df.columns:
                value_col = col
                break
        
        if not smiles_col or not value_col:
            print(f"✗ {file}: Missing SMILES or value column")
            continue
        
        print(f"Processing {file}...")
        
        # Canonical SMILES conversion
        df['canonical_smiles'] = df[smiles_col].apply(get_canonical_smiles)
        
        # Remove invalid SMILES
        df_valid = df[df['canonical_smiles'].notna()].copy()
        invalid_smiles = original_rows - len(df_valid)

        # Remove censored values: use only exact measurements with ChEMBL standard_relation '='
        # ('>', '<', '>=', '<=', '~' etc. are upper/lower bounds and unsuitable as regression labels)
        censored = 0
        if 'relation' in df_valid.columns:
            before = len(df_valid)
            df_valid = df_valid[df_valid['relation'] == '='].copy()
            censored = before - len(df_valid)
        
        # Duplicate-handling logic
        if 'doc_id' in df_valid.columns and 'assay_id' in df_valid.columns:
            # ChEMBL data: merge only true duplicates by mean
            df_result = []
            merged_count = 0
            kept_count = 0
            
            for (canonical_smiles, doc_id, assay_id), group in df_valid.groupby(
                ['canonical_smiles', 'doc_id', 'assay_id']
            ):
                if len(group) > 1:
                    # same molecule + same doc_id + same assay_id → merge by mean
                    merged_row = group.iloc[0].copy()
                    merged_row[value_col] = pd.to_numeric(group[value_col], errors='coerce').mean()
                    df_result.append(merged_row)
                    merged_count += len(group) - 1
                else:
                    # unique rows → keep as is
                    df_result.append(group.iloc[0])
                    kept_count += 1
            
            df_deduplicated = pd.DataFrame(df_result)
        else:
            # TDC data: merge by mean on canonical SMILES
            df_result = []
            merged_count = 0
            kept_count = 0
            
            for canonical_smiles, group in df_valid.groupby('canonical_smiles'):
                if len(group) > 1:
                    # same molecule → merge by mean
                    merged_row = group.iloc[0].copy()
                    merged_row[value_col] = pd.to_numeric(group[value_col], errors='coerce').mean()
                    df_result.append(merged_row)
                    merged_count += len(group) - 1
                else:
                    # unique rows → keep as is
                    df_result.append(group.iloc[0])
                    kept_count += 1
            
            df_deduplicated = pd.DataFrame(df_result)
        
        # Replace the original column with canonical_smiles
        df_deduplicated[smiles_col] = df_deduplicated['canonical_smiles']
        df_deduplicated = df_deduplicated.drop('canonical_smiles', axis=1)
        
        final_rows = len(df_deduplicated)
        
        # Save the file
        output_path = os.path.join(output_dir, file.replace('.csv', '_smart_deduplicated.csv'))
        df_deduplicated.to_csv(output_path, index=False)
        
        summary_data.append({
            'File': file,
            'Original': original_rows,
            'Invalid SMILES': invalid_smiles,
            'Censored': censored,
            'Merged': merged_count,
            'Final': final_rows,
            'Removed %': f"{(original_rows - final_rows) / original_rows * 100:.2f}%"
        })
        
        print(f"  ✓ {original_rows} → {final_rows} rows (censored {censored}, merged {merged_count})\n")
        
    except Exception as e:
        print(f"✗ {file}: Error - {e}\n")

Processing chembl_CL_raw.csv...
  ✓ 80447 → 70202 rows (censored 9733, merged 512)

Processing chembl_Fu_raw.csv...
  ✓ 14033 → 13354 rows (censored 658, merged 21)

✗ biogen_CL_raw.csv: Missing SMILES or value column
Processing chembl_PPB_raw.csv...
  ✓ 13778 → 13751 rows (censored 0, merged 27)

Processing chembl_T1_2_raw.csv...
  ✓ 94637 → 79483 rows (censored 14439, merged 715)

Processing tdc_cl_hep.csv...
  ✓ 1213 → 1020 rows (censored 0, merged 193)

Processing tdc_cl_mic.csv...
  ✓ 1102 → 1102 rows (censored 0, merged 0)

Processing tdc_half_life.csv...
  ✓ 667 → 665 rows (censored 0, merged 2)

Processing tdc_PPBR_AZ.csv...
  ✓ 1614 → 1614 rows (censored 0, merged 0)



## Results summary

In [3]:
summary_df = pd.DataFrame(summary_data)

print("\n" + "="*120)
print("Smart Deduplication Summary")
print("="*120)
print(summary_df.to_string(index=False))

print("\n" + "="*120)
print(f"Output Directory: {output_dir}")
print("="*120)

# Overall statistics
total_original = summary_df['Original'].sum()
total_invalid = summary_df['Invalid SMILES'].sum()
total_merged = summary_df['Merged'].sum()
total_final = summary_df['Final'].sum()
total_removed = total_original - total_final

print(f"\nOverall statistics:")
print(f"  Original rows: {total_original:,}")
print(f"  Invalid SMILES: {total_invalid:,}")
print(f"  Merged rows: {total_merged:,}")
print(f"  Final rows: {total_final:,}")
print(f"  Removed rows: {total_removed:,} ({total_removed/total_original*100:.2f}%)")
print(f"\nKey benefits:")
print(f"  ✓ Only true duplicates removed (same doc_id+assay_id)")
print(f"  ✓ All independent measurements kept (different assays)")
print(f"  ✓ Less noise from deduplication")
print(f"  ✓ Model strengthened by diverse measurements")


Smart Deduplication Summary
               File  Original  Invalid SMILES  Censored  Merged  Final Removed %
  chembl_CL_raw.csv     80447               0      9733     512  70202    12.74%
  chembl_Fu_raw.csv     14033               0       658      21  13354     4.84%
 chembl_PPB_raw.csv     13778               0         0      27  13751     0.20%
chembl_T1_2_raw.csv     94637               0     14439     715  79483    16.01%
     tdc_cl_hep.csv      1213               0         0     193   1020    15.91%
     tdc_cl_mic.csv      1102               0         0       0   1102     0.00%
  tdc_half_life.csv       667               0         0       2    665     0.30%
    tdc_PPBR_AZ.csv      1614               0         0       0   1614     0.00%

Output Directory: ..\data\in_vitro_data_deduplicated

Overall statistics:
  Original rows: 207,491
  Invalid SMILES: 0
  Merged rows: 1,470
  Final rows: 181,191
  Removed rows: 26,300 (12.68%)

Key benefits:
  ✓ Only true duplicates removed